#### **Problem Statement**

- Retrieval Augmented Generation is a prompting technique for optimizing the output of a large language model, by - referencing a knowledge source in addition to the training data sources before a response is generated.

- Why is Retrieval Augmented Generation important? : Improves **factual consistency** in response and making the response more trustworthy, addressing the problem of hallucination.

- Without retrieval augmented generation, a large language model takes a user input and provides a response, based on the information the model has been trained on. With retrieval augmented generation, an information retrieval component is combined with the text generation model.

- The large language model hence uses the external data source combined with the information the model has been trained on to provide a better response.

#### **References**

- https://www.promptingguide.ai/techniques/rag

- Lewis, Patrick, et al. "Retrieval-augmented generation for knowledge-intensive nlp tasks." Advances in Neural Information Processing Systems 33 (2020): 9459-9474.

- https://aws.amazon.com/what-is/retrieval-augmented-generation/

- https://www.kaggle.com/datasets/radek1/sci-or-not-sci-hypthesis-testing-pack

- https://python.langchain.com/docs/integrations/document_loaders/arxiv

- https://python.langchain.com/docs/integrations/document_loaders/merge_doc

- https://deci.ai/

- Reading to explore : ToT game of 24
https://blog.devgenius.io/tree-of-thoughts-implementation-to-solve-the-game-of-24-13fb1fa2034b

#### **Installing required libraries**

In [ ]:
# Installing dependencies : Langchain - Framework, Faiss vector database and tiktoken to work with OpenAI model
# Pinned versions (tested Oct 2026) so this notebook keeps working on Google Colab
!pip install -q langchain==0.3.27 langchain-community==0.3.27 langchain-core==0.3.72 langchain-text-splitters==0.3.9 langchain-openai==0.3.28 openai==1.99.9 tiktoken==0.14.0 arxiv==4.0.1 pymupdf==1.28.2 faiss-cpu==1.15.1
# !pip install langchain -q   # (replaced by the pinned install above)
# !pip install langchain-openai -q   # (replaced by the pinned install above)
# !pip install tiktoken -q   # (replaced by the pinned install above)
# !pip install arxiv -q   # (replaced by the pinned install above)
# !pip install langchain_community -q   # (replaced by the pinned install above)
# !pip install pymupdf -q   # (replaced by the pinned install above)
# !pip install faiss-cpu -q   # (replaced by the pinned install above)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.9/815.9 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 28.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 32.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 7.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 9.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 7.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This be

#### **Importing required libraries**

In [ ]:
# Importing required dependencies
from langchain_community.document_loaders import ArxivLoader # Importing document loader to retrieve metadata of articles published in arxiv
from langchain_community.document_loaders import MergedDataLoader # Merge documents returned by specific data loaders
from langchain_community.vectorstores import FAISS # FAISS vector database
from langchain_openai import OpenAIEmbeddings # OpenAI Embeddings
from langchain_openai import ChatOpenAI as OpenAI  # completion model retired; chat model used instead # OpenAI Interface from langchain_openai
from langchain.chains import ConversationalRetrievalChain # Importing conversational retrieval chain
from langchain.text_splitter import RecursiveCharacterTextSplitter # Text splitter
from google.colab import userdata # Importing OpenAI API Key with in the google colab environment

In [ ]:
# Extracting metadata for the articles shown in the following list

docs = ["1706.03762", "2005.11401" , "2202.01110", "2210.11610", "2005.14165", "2305.10601", "2012.15723",
        "2203.02155", "1904.04232", "2312.11985", "2304.01852"]

#### **Document Loader in Langchain**

- Document loaders : Load the data from a specific source in the form of a document
(a document consists of text with associated metadata)
- Example: TextLoader, ArxivLoader, PyPDFLoader and more.


- https://python.langchain.com/docs/modules/data_connection/document_loaders/

In [ ]:
# Loading scientific articles in a document format using ArxivLoader()

doc = ArxivLoader(query = docs[0], load_all_available_metadata=True).load()
doc[0].metadata

{'Published': '2023-08-02',
 'Title': 'Attention Is All You Need',
 'Authors': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin',
 'Summary': 'The dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks in an encoder-decoder configuration. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer, based\nsolely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to be\nsuperior in quality while being more parallelizable and requiring significantly\nless time to train. Our model achieves 28.4 BLEU on the WMT 2014\nEnglish-to-German translation task, improving over the existing best results,\nincluding ensembles by over 2 BLEU. On the WMT 2014 English-to-French\ntranslation task, 

In [ ]:
# Extracting meta-information of the scientific articles

data = []
for i in range(len(docs)):
  doc = ArxivLoader(query = docs[i]).load()
  data.append(doc[0].metadata)

In [ ]:
# list containing metadata of articles extracted using Arxiv Loader
data

[{'Published': '2023-08-02',
  'Title': 'Attention Is All You Need',
  'Authors': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, Illia Polosukhin',
  'Summary': 'The dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks in an encoder-decoder configuration. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer, based\nsolely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to be\nsuperior in quality while being more parallelizable and requiring significantly\nless time to train. Our model achieves 28.4 BLEU on the WMT 2014\nEnglish-to-German translation task, improving over the existing best results,\nincluding ensembles by over 2 BLEU. On the WMT 2014 English-to-French\ntranslation ta

#### **OpenAI API Key**

In [ ]:
# Extracting OPENAI_API_KEY using userdata.get()

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [ ]:
# Gathering the list of documents using Arxiv Loader
# https://deci.ai/
data = []
for doc in docs:
  loader = ArxivLoader(query = doc)
  data.append(loader)

In [ ]:
# Merging the documents together using MergedDataLoader()

loader = MergedDataLoader(loaders = data)
doc = loader.load()

#### **Text Splitter**

- Recursive character text split : splits text by recursively looking at characters.

- https://api.python.langchain.com/en/latest/text_splitter/langchain.text_splitter.RecursiveCharacterTextSplitter.html

In [ ]:
# Recursive Character Text Splitter with chunk size : 500 and chunk overlap 20
# creating meaninful chunks of text with some overlap for ensuring context between the chunks.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 20,
    length_function = len
)
text_split = text_splitter.split_documents(doc)

#### **Embeddings**


- The data is broken into meaningful chunks. These need to be converted into a vector representation, which involves transforming the text data into embeddings which are numeric representations that capture the semantic meaning behind text.

- Examples : OpenAI Embeddings, HuggingFaceEmbeddings with specific models such as BERT.

- https://www.datacamp.com/blog/what-is-retrieval-augmented-generation-rag

- https://python.langchain.com/docs/integrations/text_embedding/openai

In [ ]:
# Initializing OpenAIEmbeddings() class

embeddings = OpenAIEmbeddings(openai_api_key = OPENAI_API_KEY)

#### **Vector Database**

- FAISS is a library for efficient similarity search and clustering of dense vectors

- https://python.langchain.com/docs/integrations/vectorstores/faiss

In [ ]:
# FAISS.from_documents() method takes in text chunks with the appropriate embeddings model
db = FAISS.from_documents(text_split, embeddings)

#### **Retriever Interface**

- VectorStoreRetriever, backed by a vectorstore.
- When a query enters a system, it must also be converted into embeddings. The same model must be used for document and query embedding to ensure uniformity between the two.
- Two search types to retrieve text : Retrieve text based on similarity : uses similarity search in the retriever object, where it selects the text chunk vectors that are most similar (using cosine similarity or Euclidean distance) to the question vector.
- "MMR", maximum marginal relevance search, optimizing for similarity to query AND diversity among the selected documents.

- https://towardsdatascience.com/4-ways-of-question-answering-in-langchain-188c6707cc5


In [ ]:
# Vector Store Retriever
retriever = db.as_retriever(search_type ="similarity", search_kwargs = {"k":5})

#### **Creating a chain for QA**

- Uses load_qa_chain, provides a generic interface for answering questions.

- Retrieval QA Chain, retrieves the most relevant chunk of text and feeds those to the language model.

- Conversational Retrieval Chain takes chat history for follow up questions.

- ConversationalRetrievalChain = RetrievalQAChain + conversation Memory

- https://api.python.langchain.com/en/latest/chains/langchain.chains.conversational_retrieval.base.ConversationalRetrievalChain.html

In [ ]:
# ConversationalRetrievalChain.from_llm() method with the gpt-4o-mini model

llm = OpenAI(openai_api_key = OPENAI_API_KEY, model = 'gpt-4o-mini')
chain = ConversationalRetrievalChain.from_llm(llm = llm, retriever = retriever)
chat_history = []

In [ ]:
# Query for the ConversationalRetrivalChain()

query = "Where could large language models fall short according to `Tree of Thoughts: Deliberate Problem Solving with Large Language Models`?"

In [ ]:
# Extracting the response from the model
response = chain({"question": query, "chat_history" : chat_history})
response['answer']

' Large language models could fall short in tasks that require exploration, strategic lookahead, or where initial decisions play a pivotal role, according to "Tree of Thoughts: Deliberate Problem Solving with Large Language Models." '